In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from neo4j import GraphDatabase


In [2]:
raw_directory = '../data/raw'

In [3]:
euroscivoc = pd.read_excel(raw_directory + "/euroSciVoc" + ".xlsx")
legalbasis = pd.read_excel(raw_directory + "/legalBasis" + ".xlsx")
organization = pd.read_excel(raw_directory + "/organization" + ".xlsx")
project = pd.read_excel(raw_directory + "/project" + ".xlsx")
deliverables = pd.read_excel(raw_directory + "/projectDeliverables" + ".xlsx")
publications = pd.read_excel(raw_directory + "/projectPublications" + ".xlsx")
reports = pd.read_excel(raw_directory + "/reportSummaries" + ".xlsx")
topics = pd.read_excel(raw_directory + "/topics" + ".xlsx")
weblink = pd.read_excel(raw_directory + "/webLink" + ".xlsx")

/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/open

In [4]:
# Extract latitude and longitude from geolocation column
# Assumes geolocation is a string like 'lat,lon' or a tuple/list

def extract_lat_lon(geo):
    if pd.isnull(geo):
        return pd.Series({'latitude': None, 'longitude': None})
    if isinstance(geo, str):
        try:
            lat, lon = map(float, geo.split(','))
            return pd.Series({'latitude': lat, 'longitude': lon})
        except Exception:
            return pd.Series({'latitude': None, 'longitude': None})
    if isinstance(geo, (list, tuple)) and len(geo) == 2:
        return pd.Series({'latitude': geo[0], 'longitude': geo[1]})
    return pd.Series({'latitude': None, 'longitude': None})

organization[['latitude', 'longitude']] = organization['geolocation'].apply(extract_lat_lon)

# Select relevant columns and aggregate ecContribution
org_cols = ['organisationID', 'name', 'country', 'latitude', 'longitude']
organization_nodes = organization[org_cols].copy()
organization_nodes.head()

,organisationID,name,country,latitude,longitude
0,999981634,WAGENINGEN UNIVERSITY,NL,51.986328,5.667937
1,999997736,AARHUS UNIVERSITET,DK,56.171028,10.199381
2,999854855,UNIVERSITAET POTSDAM,DE,52.397917,13.014627
3,999990267,MAX-PLANCK-GESELLSCHAFT ZUR FORDERUNG DER WISS...,DE,48.141169,11.582293
4,999874546,UNIVERSIDAD COMPLUTENSE DE MADRID,ES,40.434340,-3.734064


In [5]:
def extract_topics_from_path(df, path_col, id_col):
    # Remove leading/trailing slashes, split by '/', and explode
    df = df[[id_col, path_col]].copy()
    df[path_col] = df[path_col].fillna('').apply(lambda x: x.strip('/'))
    df['topic'] = df[path_col].apply(lambda x: x.split('/') if x else [])
    df = df.explode('topic')
    df = df[[id_col, 'topic']]
    df = df[df['topic'].str.strip() != '']
    df['topic'] = df['topic'].str.strip()
    return df.reset_index(drop=True)

project_topics = extract_topics_from_path(euroscivoc, 'euroSciVocPath', 'projectID')
project_topics.rename(columns={'projectID': 'project_id'}, inplace=True)
project_topics.head(10)

,project_id,topic
0,101116741,social sciences
1,101116741,political sciences
2,101116741,government systems
3,101163161,agricultural sciences
4,101163161,"agriculture, forestry, and fisheries"
5,101163161,agriculture
6,101163161,grains and oilseeds
7,101163161,natural sciences
8,101163161,physical sciences
9,101163161,optics


In [7]:
# Merge project_topics with organization to get organization info for each project-topic
# Assumes organization has columns: organisationID, name, country, latitude, longitude, ecContribution, projectID

# First, join project_topics with organization on project_id/projectID
merged = project_topics.merge(organization, left_on='project_id', right_on='projectID', how='left')



# Rename columns for clarity
agg.rename(columns={'name': 'organizationName'}, inplace=True)

# Show the result
agg.head(10)

,project_id,topic,projectID,projectAcronym,organisationID,vatNumber,name,shortName,SME,activityType,...,rcn,order,role,ecContribution,netEcContribution,totalCost,endOfParticipation,active,latitude,longitude
0,101116741,social sciences,101116741,DOE,999981634,NL811383696B01,WAGENINGEN UNIVERSITY,WU,False,HES,...,1906458,1,coordinator,1499998.0,1499998.0,1499998,False,NaN,51.986328,5.667937
1,101116741,political sciences,101116741,DOE,999981634,NL811383696B01,WAGENINGEN UNIVERSITY,WU,False,HES,...,1906458,1,coordinator,1499998.0,1499998.0,1499998,False,NaN,51.986328,5.667937
2,101116741,government systems,101116741,DOE,999981634,NL811383696B01,WAGENINGEN UNIVERSITY,WU,False,HES,...,1906458,1,coordinator,1499998.0,1499998.0,1499998,False,NaN,51.986328,5.667937
3,101163161,agricultural sciences,101163161,IRASTRO,999997736,DK31119103,AARHUS UNIVERSITET,AU,False,HES,...,1905956,2,participant,4246240.0,4246240.0,4246240,False,NaN,56.171028,10.199381
4,101163161,agricultural sciences,101163161,IRASTRO,999854855,DE138408327,UNIVERSITAET POTSDAM,UP,False,HES,...,1908936,3,participant,1982813.0,1982813.0,1982813,False,NaN,52.397917,13.014627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923118,101114240,psychiatry,101114240,MindGuide,889095017,SI18700187,ZAMAX STORITVE DOO,ZAMAX,True,PRC,...,1981670,1,coordinator,75000.0,75000.0,0,False,NaN,46.042573,14.502807
923119,101113949,medical and health sciences,101113949,orthomobile,884816250,TR5421342866,KEDI MOBIL UYGULAMA ANONIM SIRKETI,NaN,True,PRC,...,1983489,1,coordinator,75000.0,75000.0,0,False,NaN,37.161431,28.375754
923120,101113949,clinical medicine,101113949,orthomobile,884816250,TR5421342866,KEDI MOBIL UYGULAMA ANONIM SIRKETI,NaN,True,PRC,...,1983489,1,coordinator,75000.0,75000.0,0,False,NaN,37.161431,28.375754
923121,101113949,odontology,101113949,orthomobile,884816250,TR5421342866,KEDI MOBIL UYGULAMA ANONIM SIRKETI,NaN,True,PRC,...,1983489,1,coordinator,75000.0,75000.0,0,False,NaN,37.161431,28.375754


In [19]:
# Save the aggregated results to CSV
agg.to_csv('../data/processed/org_by_research.csv', index=False)
print("Saved results to '../data/processed/org_by_research.csv'")

Saved results to '../data/processed/org_by_research.csv'
